# ⚡ Fluxos de Trabalho de Agentes Concorrentes com Microsoft Foundry (Python)

## 📋 Tutorial Avançado de Processamento em Paralelo

Este notebook demonstra **padrões de fluxo de trabalho concorrentes** usando o Microsoft Agent Framework. Vai aprender a construir fluxos de trabalho de alto desempenho e processamento paralelo onde múltiplos agentes de IA executam simultaneamente, melhorando drasticamente o rendimento e permitindo processos de negócio multi-threaded sofisticados.

> **Nota de migração:** Este exemplo referenciava anteriormente os Modelos do GitHub. Os Modelos do GitHub estão descontinuados (fim em julho de 2026), portanto agora usa **Microsoft Foundry** através do `FoundryChatClient`, que aponta para a API de **Respostas** do Azure OpenAI.

## 🎯 Objetivos de Aprendizagem

### 🚀 **Fundamentos do Processamento Concorrente**
- **Execução Paralela de Agentes**: Execute múltiplos agentes simultaneamente para máximo rendimento
- **Orquestração de Fluxos de Trabalho**: Coordene operações concorrentes mantendo a consistência dos dados
- **Otimização de Performance**: Alcance aceleração significativa através do processamento paralelo
- **Gestão de Recursos**: Utilize eficientemente os recursos dos modelos de IA nas operações concorrentes

### 🏗️ **Padrões Avançados de Concorrência**
- **Processamento Fork-Join**: Divida o trabalho entre vários agentes e una os resultados
- **Paralelismo em Pipeline**: Sobreposição de etapas de execução para rendimento contínuo
- **Balanceamento de Carga**: Distribua o trabalho de forma equilibrada pelos agentes disponíveis
- **Pontos de Sincronização**: Coordene agentes concorrentes em etapas críticas do fluxo de trabalho

### 🏢 **Aplicações Concorrentes Empresariais**
- **Processamento de Documentos em Alto Volume**: Processar múltiplos documentos simultaneamente
- **Análise de Conteúdo em Tempo Real**: Análise concorrente de fluxos de dados recebidos
- **Otimização de Processamento em Lote**: Maximize o rendimento para operações em larga escala
- **Análise Multi-Modal**: Processamento paralelo de diferentes tipos de conteúdo (texto, imagens, dados)

## ⚙️ Pré-requisitos & Configuração

### 📦 **Dependências Necessárias**

Instale o Agent Framework com capacidades de fluxo de trabalho concorrente:

```bash
pip install agent-framework -U
```

### 🔑 **Configuração do Microsoft Foundry**

Inicie sessão com o Azure CLI (`az login`) para que o `AzureCliCredential` possa autenticar, depois defina os detalhes do seu projeto Microsoft Foundry.

**Configuração do Ambiente (ficheiro .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

**Considerações sobre Processamento Concorrente:**
- **Limites de Taxa**: Monitorize os limites de taxa da Azure OpenAI para pedidos concorrentes
- **Uso de Recursos**: Considere o uso de memória e CPU com múltiplos agentes concorrentes
- **Gestão de Erros**: Implemente recuperação robusta de erros para operações paralelas

### 🏗️ **Arquitetura de Fluxos de Trabalho Concorrentes**

```mermaid
graph TD
    A[Início do Fluxo de Trabalho] --> B[Execução Concorrente]
    B --> C[Conjunto de Agentes 1]
    B --> D[Conjunto de Agentes 2]
    B --> E[Conjunto de Agentes 3]
    C --> F[Agregação de Resultados]
    D --> F
    E --> F
    F --> G[Resultado Final]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Benefícios Principais:**
- **⚡ Performance**: Aceleração significativa através da execução paralela
- **📈 Escalabilidade**: Gerir cargas crescentes sem aumento proporcional de tempo
- **🔄 Eficiência**: Melhor aproveitamento dos recursos computacionais disponíveis
- **🎯 Rendimento**: Processar mais trabalho no mesmo intervalo de tempo

## 🎨 **Padrões de Design de Fluxos de Trabalho Concorrentes**

### 🔍 **Pipeline de Investigação & Análise**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Fluxo de Trabalho de Processamento de Dados**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Pipeline de Criação de Conteúdo**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Processamento Multi-Etapa**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Benefícios Empresariais de Performance**

### ⚡ **Otimização de Rendimento**
- **Execução Paralela**: Vários agentes a trabalhar simultaneamente
- **Utilização de Recursos**: Máxima eficiência da capacidade dos modelos de IA
- **Redução de Tempo**: Diminuição significativa do tempo total de processamento
- **Arquitetura Escalável**: Fácil adição de mais agentes concorrentes conforme necessário

### 🛡️ **Confiabilidade & Resiliência**
- **Tolerância a Falhas**: Falhas em agentes individuais não interrompem o fluxo todo
- **Isolamento de Erros**: Problemas numa ramificação concorrente não afetam as outras
- **Degradação Elegante**: O sistema continua a operar mesmo com capacidade reduzida de agentes
- **Mecanismos de Recuperação**: Repetição automática e gestão de erros para operações falhadas

### 📊 **Monitorização & Observabilidade**
- **Rastreio de Execução Concorrente**: Monitorize o progresso de todas as operações paralelas
- **Métricas de Performance**: Meça o ganho de velocidade e eficiência
- **Análise de Uso de Recursos**: Otimize a alocação dos agentes concorrentes
- **Identificação de Gargalos**: Localize e resolva restrições de performance

Vamos construir fluxos de trabalho de IA concorrentes de alto desempenho! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
